In [ ]:
import requests
import os
import base64
import json
from bs4 import BeautifulSoup
import time

# **載入client_id與client_secret有兩種方式：**


*   在本地端，用dotenv載入環境變數client_id與client_secret
*   用input手動輸入，但id&secret就是programmer自己記著



# **在本地端輸入client_id與client_secret可以用dotenv載入環境變數**


In [ ]:
# from dotenv import load_dotenv

# load_dotenv()  # 如果沒有要用os.getenv() function的話，不一定要寫這個
# client_id = os.getenv('CLIENT_ID') #'CLIENT_ID'4應該另寫於名為.env檔、並且跟本地端的.py檔放在同個資料夾，os.getenv就會自己去抓到.env檔的資料
# client_secret = os.getenv('CLIENT_SECRET') #'CLIENT_SECRET'應該另寫於名為.env檔、並且跟本地端的.py檔放在同個資料夾，os.getenv就會自己去抓到.env檔的資料

# **若不預先存放client_id與client_secret，則可以用input手動輸入。**

In [ ]:
client_id = input('請輸入client_id')

In [ ]:
client_secret = input('請輸入client_secret')

# **定義函式取得API token**

In [ ]:
# 向Spotify取得token,以利後面連進API server


def get_token():
    """根據Spotify Web Server (https://developer.spotify.com/documentation/web-api/tutorials/getting-started)
    寫的指引，須使用request.post函式來取得token，其中auth_base64、url、headers、data都是根據官方指定的寫法"""

    auth_string = client_id + ":" + client_secret
    auth_bytes = auth_string.encode('utf-8')
    auth_base64 = str(base64.b64encode(auth_bytes), 'utf-8')

    url = 'https://accounts.spotify.com/api/token'
    headers = {'Authorization': 'Basic ' + auth_base64,
               'Content-Type': 'application/x-www-form-urlencoded'}
    data = {'grant_type': 'client_credentials'}
    result = requests.post(url, headers=headers, data=data)

    # 官方文件有說回傳的response物件是屬於json格式。所以使用json.loads()轉換成python dictionary
    json_result = json.loads(result.content)
    # 轉成dictionary後，裡面會有三個key: access_token、token_type、expire_in
    # expire_in一般是3600秒（一小時）
    token = json_result['access_token']  # 字典取值
    return token

# **定義函式將token組合出官方要求的headers寫法**

In [ ]:
def get_auther_header(token):
    """根據官方文件，稍後向API server request資料時，headers參數一定寫成下方那樣"""
    return {'Authorization': 'Bearer ' + token}

In [ ]:
# 單一歌曲搜索


def get_spotify_data_track(url, token, sleeptime=3):
    headers = get_auther_header(token)
    response = requests.get(url, headers=headers)

    # 小睡一下，很重要，避免被ban
    time.sleep(sleeptime)

    if response.status_code != 200:
        print('Request failed', response.status_code, response.text)
        return None
    else:
        return response
